In [ ]:
import pandas as pd

dataset_url = "https://raw.githubusercontent.com/sravanioffice1997-arch/Placement-Readiness-Intelligence-System/main//Source/placement_readiness_dataset.csv"
df = pd.read_csv(dataset_url)

print("Shape:", df.shape)
df.head()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# ---- Load the dataset from Step 1 ----
# df = pd.read_csv("placement_readiness_dataset.csv") # This line is removed

feature_cols = [c for c in df.columns if c != "readiness_label"]
X = df[feature_cols]
y = df["readiness_label"]

# ---- Encode labels (models need numbers, not strings) ----
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# ---- Train/test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# ---- Define the 4 candidate models ----
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42),
}

# ---- Train each, evaluate on test set ----
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    trained_models[name] = model
    print(f"\n{'='*50}")
    print(f"{name} — Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=label_encoder.classes_))

# ---- Pick the best model ----
best_name = max(results, key=results.get)
best_model = trained_models[best_name]
print(f"\n{'='*50}")
print(f"BEST MODEL: {best_name} (accuracy = {results[best_name]:.4f})")
print(f"{'='*50}")

# ---- Confusion matrix for the winner ----
best_preds = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_preds)
print("\nConfusion Matrix ({})".format(best_name))
print(pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_))

# ---- Feature importance (if the winner supports it) ----
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)
    print("\nFeature Importances:")
    print(importance_df)

# ---- Save the winning model + label encoder + feature column order ----
joblib.dump(best_model, "readiness_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(feature_cols, "feature_columns.pkl")

print("\nSaved: readiness_model.pkl, label_encoder.pkl, feature_columns.pkl")

In [ ]:
# ============================================================
# STEP 3: Resume + JD Analyzer using Groq API (contextual, no fixed skill list)
# ============================================================

!pip install groq PyPDF2 -q

from groq import Groq
import PyPDF2
import json
import re

# ---- Hardcode your Groq API key here ----
GROQ_API_KEY = "PASTE_YOUR_GROQ_API_KEY_HERE"
client = Groq(api_key=GROQ_API_KEY)

# ---- 1. Extract text from resume PDF ----
def extract_resume_text(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text.strip()

# ---- 2. Ask Groq to analyze the resume (contextual, not keyword-based) ----
def analyze_resume(resume_text):
    prompt = f"""
You are an expert technical recruiter. Read the resume text below and extract structured information.
Do NOT rely on any fixed skill list — infer skills contextually from projects, experience, and tools mentioned.

Return ONLY valid JSON, no markdown, no extra text, in this exact structure:
{{
  "skills": ["list", "of", "all", "technical", "and", "soft", "skills", "found"],
  "projects": ["short description of each project"],
  "certifications": ["list of certifications, empty list if none"],
  "internships": ["list of internships/work experience, empty list if none"],
  "education": "highest degree and field",
  "tools_and_technologies": ["list of tools/frameworks/languages"],
  "resume_completeness_score": <integer 0-100, based on how complete/detailed the resume is>
}}

Resume text:
\"\"\"{resume_text}\"\"\"
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json|```$", "", raw, flags=re.MULTILINE).strip()
    return json.loads(raw)

# ---- 3. Ask Groq to analyze the JD ----
def analyze_jd(jd_text):
    prompt = f"""
You are an expert technical recruiter. Read the job description below and extract structured requirements.
Do NOT rely on any fixed skill list — infer required skills contextually.

Return ONLY valid JSON, no markdown, no extra text, in this exact structure:
{{
  "job_title": "string",
  "role_category": "string (e.g. Data Science, Web Development, DevOps)",
  "required_skills": ["list of all required skills"],
  "critical_skills": ["subset of required_skills that are absolutely essential"],
  "optional_skills": ["nice-to-have skills"],
  "tools_and_frameworks": ["list"],
  "soft_skills": ["list"],
  "responsibilities": ["list of key responsibilities"]
}}

Job description text:
\"\"\"{jd_text}\"\"\"
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json|```$", "", raw, flags=re.MULTILINE).strip()
    return json.loads(raw)

# ---- 4. Skill Gap Analyzer ----
def analyze_skill_gap(resume_data, jd_data):
    prompt = f"""
Compare the candidate's resume profile with the job requirements below.
Match skills semantically (e.g. "React.js" matches "React", "ML" matches "Machine Learning") — don't require exact string matches.

Resume skills: {resume_data['skills']}
Resume tools: {resume_data['tools_and_technologies']}
JD required skills: {jd_data['required_skills']}
JD critical skills: {jd_data['critical_skills']}

Return ONLY valid JSON, no markdown, in this exact structure:
{{
  "matched_skills": ["skills present in both"],
  "missing_skills": ["required skills not found in resume"],
  "critical_missing_skills": ["critical skills not found in resume"],
  "skills_to_learn_first": ["top 3-5 priority skills to learn"]
}}
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json|```$", "", raw, flags=re.MULTILINE).strip()
    return json.loads(raw)

print("Step 3 functions loaded: extract_resume_text, analyze_resume, analyze_jd, analyze_skill_gap")

In [ ]:
# ---- TEST RUN (using GitHub-hosted resume, no manual upload) ----
import urllib.request

resume_url = "https://raw.githubusercontent.com/sravanioffice1997-arch/Placement-Readiness-Intelligence-System/main//Source/pathipakasravani_pdf.pdf"
resume_path = "resume.pdf"
urllib.request.urlretrieve(resume_url, resume_path)

resume_text = extract_resume_text(resume_path)

sample_jd = """
We are hiring a Data Analyst with experience in Python, SQL, Power BI, and statistics.
Familiarity with machine learning basics is a plus. Must have strong communication skills.
"""

resume_data = analyze_resume(resume_text)
jd_data = analyze_jd(sample_jd)
gap_data = analyze_skill_gap(resume_data, jd_data)

print("RESUME DATA:", json.dumps(resume_data, indent=2))
print("\nJD DATA:", json.dumps(jd_data, indent=2))
print("\nSKILL GAP:", json.dumps(gap_data, indent=2))

In [ ]:
import joblib
import numpy as np
import pandas as pd # Import pandas for DataFrame operations
import re # Import re for keyword matching

# ---- Load the trained model artifacts from Step 2 ----
model = joblib.load("readiness_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")
feature_cols = joblib.load("feature_columns.pkl")

def build_features(resume_data, jd_data, gap_data):
    """
    Converts LLM outputs into the 10 numeric features.
    RULE: all percentage/match scores are computed relative to the JD's
    requirements (denominator = JD), never relative to the resume's total
    skill count. Extra resume skills not asked for by the JD do NOT inflate
    the score.
    """

    required = jd_data.get("required_skills", []) or []
    critical = jd_data.get("critical_skills", []) or []
    optional_skills_jd = jd_data.get("optional_skills", []) or [] # Added this line
    matched = gap_data.get("matched_skills", []) or []
    missing = gap_data.get("missing_skills", []) or []
    critical_missing = gap_data.get("critical_missing_skills", []) or []

    # ---- skill_match_percentage: matched / JD-required (JD-anchored) ----
    skill_match_percentage = round((len(matched) / len(required)) * 100) if required else 0

    # ---- critical_skill_match_percentage ----
    critical_matched = [s for s in matched if s in critical]
    critical_skill_match_percentage = round((len(critical_matched) / len(critical)) * 100) if critical else 0

    # ---- missing_skills_count ----
    missing_skills_count = len(missing)

    # ---- critical_missing_skills_count ----
    critical_missing_skills_count = len(critical_missing)

    # ---- optional_skill_match_percentage ---- # Added this block
    resume_skills_set = set(resume_data.get("skills", []) or [])
    optional_matched_skills = [s for s in optional_skills_jd if s in resume_skills_set]
    optional_skill_match_percentage = round((len(optional_matched_skills) / len(optional_skills_jd)) * 100) if optional_skills_jd else 0

    # ---- project_relevance_score: needs to be derived from LLM's understanding ----
    # For simplicity, let's assume a dummy value or a direct extraction if available.
    # In a real system, the LLM would provide a specific score for this.
    # For now, let's use a placeholder or derive from resume completeness.
    project_relevance_score = resume_data.get("resume_completeness_score", 0)

    # ---- certification_relevance_score ----
    certification_relevance_score = len(resume_data.get("certifications", [])) * 20 # Dummy scoring
    certification_relevance_score = min(certification_relevance_score, 100)

    # ---- internship_relevance_score ----
    internship_relevance_score = len(resume_data.get("internships", [])) * 30 # Dummy scoring
    internship_relevance_score = min(internship_relevance_score, 100)

    # ---- resume_completeness_score ----
    resume_completeness_score = resume_data.get("resume_completeness_score", 0)

    # ---- keyword_match_score: a proxy for how many distinct JD words appear in resume ----
    jd_words = set(re.findall(r'\b\w+\b', jd_data.get("job_title", "").lower() + " " + " ".join(jd_data.get("required_skills", [])).lower()))
    resume_words = set(re.findall(r'\b\w+\b', " ".join(resume_data.get("skills", [])).lower() + " " + " ".join(resume_data.get("projects", [])).lower()))
    keyword_matches = len(jd_words.intersection(resume_words))
    keyword_match_score = round((keyword_matches / len(jd_words)) * 100) if jd_words else 0

    # ---- role_category_match_score: simple 100 if match, 0 if not ----
    # This assumes 'role_category' in JD and LLM can infer category for resume
    # For now, let's just check if JD has a category and if it's broadly related to data science
    jd_role_category = jd_data.get("role_category", "").lower()
    if jd_role_category and "data science" in jd_role_category or "data analyst" in jd_role_category:
      role_category_match_score = 100 # Assuming the resume is data-related
    else:
      role_category_match_score = 0

    features = [
        skill_match_percentage,
        critical_skill_match_percentage,
        missing_skills_count,
        critical_missing_skills_count,
        optional_skill_match_percentage, # Added this line
        project_relevance_score,
        certification_relevance_score,
        internship_relevance_score,
        resume_completeness_score,
        keyword_match_score,
        role_category_match_score,
    ]

    # Ensure features are in the correct order as expected by the model
    feature_dict = dict(zip(feature_cols, features))
    # Create a DataFrame with a single row for prediction
    input_df = pd.DataFrame([feature_dict])

    return input_df[feature_cols] # Return in the exact order the model expects

def predict_readiness(resume_data, jd_data, gap_data):
    input_features_df = build_features(resume_data, jd_data, gap_data)
    prediction_encoded = model.predict(input_features_df)[0]
    readiness_label = label_encoder.inverse_transform([prediction_encoded])[0]

    # Get class probabilities
    probabilities = model.predict_proba(input_features_df)[0]
    class_probabilities = {label: prob for label, prob in zip(label_encoder.classes_, probabilities)}

    # For readiness_score, we can use the probability of the predicted class, scaled to 100
    readiness_score = round(class_probabilities[readiness_label] * 100)

    # Convert input_features_df to a dictionary for easier printing/display
    features_dict = input_features_df.iloc[0].to_dict()

    return {
        "features": features_dict,
        "readiness_score": readiness_score,
        "readiness_label": readiness_label,
        "class_probabilities": class_probabilities,
    }

print("Step 4 functions loaded: build_features, predict_readiness")

In [ ]:
# ---- Standalone test using your actual Step 3 output ----
resume_data = {
  "skills": ["Data Science","Machine Learning","Data Analysis","Data Visualization","Python","SQL","Pandas","NumPy","Matplotlib","Seaborn","Scikit-Learn","Regression","Classification","Feature Engineering","Generative AI","LLMs","Prompt Engineering","Streamlit","Gradio","Git","GitHub","VS Code","Power BI","Time Series Forecasting","Natural Language Processing","Deep Learning","XGBoost","Random Forest","Team Management","Communication"],
  "projects": ["AI-powered resume analyzer using Python & Gemini API","Financial news sentiment analysis model and LLM placement analyzer","Healthcare no-show prediction and risk clustering using XGBoost and Random Forest","Delhi Metro travel patterns and public distribution systems analysis using SQL and analytics"],
  "certifications": ["Master Data Science Program"],
  "internships": [],
  "education": "MBA Human Resources (HR) and B.Tech Electronics and Instrumentation Engineering (EIE)",
  "tools_and_technologies": ["Python","SQL","Pandas","NumPy","Matplotlib","Seaborn","Scikit-Learn","Gemini API","Streamlit","Gradio","Git","GitHub","VS Code","Power BI","XGBoost","Random Forest"],
  "resume_completeness_score": 80
}

jd_data = {
  "job_title": "Data Analyst",
  "role_category": "Data Science",
  "required_skills": ["Python","SQL","Power BI","statistics","communication"],
  "critical_skills": ["Python","SQL","statistics","communication"],
  "optional_skills": ["machine learning"],
  "tools_and_frameworks": ["Power BI"],
  "soft_skills": ["communication"],
  "responsibilities": ["data analysis"]
}

gap_data = {
  "matched_skills": ["Python","SQL","Power BI","communication","statistics"],
  "missing_skills": [],
  "critical_missing_skills": [],
  "skills_to_learn_first": []
}

# ---- Run prediction ----
result = predict_readiness(resume_data, jd_data, gap_data)

print("FEATURES BUILT:")
for k, v in result["features"].items():
    print(f"  {k}: {v}")

print(f"\nPlacement Readiness Score: {result['readiness_score']} / 100")
print(f"Readiness Level: {result['readiness_label']}")
print(f"\nClass Probabilities:")
for label, prob in result["class_probabilities"].items():
    print(f"  {label}: {prob*100:.1f}%")

In [ ]:
# ============================================================
# STEP 5: LLM-Powered Feedback & Improvement Plan Generator
# ============================================================

def generate_feedback(resume_data, jd_data, gap_data, prediction_result):
    """
    Sends the full picture (resume, JD, skill gap, ML prediction) to Groq
    and asks for a structured coaching report.
    """
    prompt = f"""
You are a career coach helping a student prepare for a job application.

Job Title: {jd_data.get('job_title')}
Role Category: {jd_data.get('role_category')}

Placement Readiness Score: {prediction_result['readiness_score']}/100
Readiness Level: {prediction_result['readiness_label']}

Matched Skills: {gap_data.get('matched_skills')}
Missing Skills: {gap_data.get('missing_skills')}
Critical Missing Skills: {gap_data.get('critical_missing_skills')}

Candidate's Projects: {resume_data.get('projects')}
Candidate's Certifications: {resume_data.get('certifications')}
Candidate's Education: {resume_data.get('education')}

Based on this, generate a coaching report. Return ONLY valid JSON, no markdown, in this exact structure:
{{
  "summary": "10-20 line easy-to-read summary explaining whether this student is suitable for this job role, written directly to the student in second person",
  "strengths": ["list of 3-5 specific strengths based on their actual resume"],
  "gaps": ["list of specific gaps, referencing missing_skills if any, otherwise note minor improvement areas"],
  "seven_day_plan": ["list of 4-6 concrete, specific daily/short-term action items"],
  "thirty_day_plan": ["list of 4-6 concrete, longer-term action items"],
  "resume_improvement_suggestions": ["list of 3-5 specific suggestions to improve the resume itself"],
  "interview_tips": ["list of 3-5 tips specific to this role and this candidate's background"]
}}
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json|```$", "", raw, flags=re.MULTILINE).strip()
    return json.loads(raw)


# ---- Run it ----
feedback = generate_feedback(resume_data, jd_data, gap_data, result)

print("SUMMARY:\n", feedback["summary"])
print("\nSTRENGTHS:")
for s in feedback["strengths"]: print(" -", s)
print("\nGAPS:")
for g in feedback["gaps"]: print(" -", g)
print("\n7-DAY PLAN:")
for d in feedback["seven_day_plan"]: print(" -", d)
print("\n30-DAY PLAN:")
for d in feedback["thirty_day_plan"]: print(" -", d)
print("\nRESUME IMPROVEMENT SUGGESTIONS:")
for r in feedback["resume_improvement_suggestions"]: print(" -", r)
print("\nINTERVIEW TIPS:")
for t in feedback["interview_tips"]: print(" -", t)